# XLM-R — Many-to-Many (MTM, cyclic), Unweighted BCE

Trains XLM-R with the cyclic many-to-many language-rotation scheme on MultiEURLEX:
each document's assigned language shifts by one position per epoch
(`language = LANGUAGES[(index + epoch) % 23]`), so the model sees genuine
multilingual exposure while holding data volume/compute equal to the OTM baseline.
Validated on a 5-language subset (en/de/fr/bg/hu, one per language family) each
epoch, then evaluated on the full 23-language test set at the end.

In [ ]:
!pip -q install -U datasets transformers accelerate scikit-learn psutil

from collections import Counter
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_from_disk

dataset = load_from_disk("/content/drive/MyDrive/multieurlex_all_languages")

label_feature = dataset["train"].features["labels"].feature
print("Train / Val / Test sizes:",
      dataset["train"].num_rows, dataset["validation"].num_rows, dataset["test"].num_rows)
print("Total possible label classes:", len(label_feature.names))

## Configuration

In [ ]:
import os, time, random, gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from accelerate import Accelerator

accelerator = Accelerator(mixed_precision='no')  # fp32 throughout

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
THRESHOLD = 0.50
N_TRAIN_PARTS = 10          # training data is processed in chunks to limit RAM use

#All three transformer models (to be ran one at a time).
MODEL_NAME = "bert-base-uncased"               # BERT
#MODEL_NAME = "bert-base-multilingual-cased"   # mBERT
#MODEL_NAME = "xlm-roberta-base"               # XLM-R

TEXT_COLUMN = "text"
LABEL_COLUMN = "labels"

total_training_samples = len(dataset["train"])
total_steps = (total_training_samples // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)

print("Model:", MODEL_NAME)
print("Total training samples:", total_training_samples)
print("Total training steps:", total_steps)
print("Warmup steps:", warmup_steps)

## Metrics

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

def multilabel_metrics(y_true, y_pred):
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "exact_match": accuracy_score(y_true, y_pred),
    }

## Label vocabulary

In [ ]:
all_label_names = label_feature.names
NUM_LABELS = len(all_label_names)
label_to_index = {i: i for i in range(NUM_LABELS)}

print("Number of labels:", NUM_LABELS)

def multi_hot(label_ids, mapping):
    vector = np.zeros(len(mapping), dtype=np.float32)
    for label_id in label_ids:
        vector[mapping[int(label_id)]] = 1.0
    return vector

## Cyclic language rotation (MTM)

The language assigned to each document shifts by one position every epoch. Document-level translation coverage is uneven across languages, so a document falls back to its English text when its assigned language's translation isn't available.

In [ ]:
from functools import partial
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

LANGUAGES = ['bg', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 'fi', 'fr', 'hr',
             'hu', 'it', 'lt', 'lv', 'mt', 'nl', 'pl', 'pt', 'ro', 'sk', 'sl', 'sv']

# One representative language per family, used for fast multilingual per-epoch validation
VAL_LANGUAGES = ["en", "de", "fr", "bg", "hu"]

def assign_language(idx, epoch):
    # Deterministic: the same (idx, epoch) always maps to the same language.
    return LANGUAGES[(idx + epoch) % len(LANGUAGES)]

def preprocess_function_multilingual(examples, indices, tokenizer, label_to_index,
                                      max_length, offset=0, epoch=0):
    texts_to_tokenize = []
    for i, example in zip(indices, examples[TEXT_COLUMN]):
        lang = assign_language(i + offset, epoch)
        text = example.get(lang, "") or example.get("en", "")  # fallback to English if missing
        texts_to_tokenize.append(text)

    tokenized_inputs = tokenizer(
        texts_to_tokenize, truncation=True, max_length=max_length, padding=False
    )
    tokenized_inputs["labels"] = [
        multi_hot([lbl for lbl in row if int(lbl) in label_to_index], label_to_index)
        for row in examples[LABEL_COLUMN]
    ]
    return tokenized_inputs

train_split_size = len(dataset["train"])
part_size = train_split_size // N_TRAIN_PARTS
print(f"Training data processed in {N_TRAIN_PARTS} parts (approx {part_size} examples each).")

# NOTE: the map fn is rebuilt per chunk/epoch inside the training loop, since
# language assignment depends on both the chunk's starting offset and the epoch.

## Model, optimizer, scheduler

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification"
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)

print("Model:", MODEL_NAME, "| Number of labels:", model.config.num_labels)

## Multilingual evaluation function

Used both for per-epoch validation (on `VAL_LANGUAGES`) and the final full 23-language test evaluation.

In [ ]:
from tqdm.auto import tqdm

def evaluate_multilingual_full(model, tokenizer, raw_dataset, label_to_index,
                                num_labels, languages, batch_size=16, max_length=256,
                                compute_loss=True):
    model.eval()
    probs_by_lang, true_by_lang = {}, {}
    total_loss, n_batches = 0.0, 0
    loss_fn = torch.nn.BCEWithLogitsLoss() if compute_loss else None

    for lang in tqdm(languages, desc="Languages", leave=False):
        texts, label_rows = [], []
        for row in raw_dataset:
            text = row["text"].get(lang)
            if text:
                texts.append(text)
                mh = np.zeros(num_labels, dtype=np.float32)
                for l in row["labels"]:
                    if l in label_to_index:
                        mh[label_to_index[l]] = 1.0
                label_rows.append(mh)
        if not texts:
            continue

        logits_list = []
        for i in tqdm(range(0, len(texts), batch_size), desc=f"  {lang}", leave=False):
            batch_texts = texts[i:i+batch_size]
            batch_labels = np.stack(label_rows[i:i+batch_size])
            enc = tokenizer(batch_texts, truncation=True, padding=True,
                             max_length=max_length, return_tensors="pt").to(accelerator.device)
            with torch.no_grad():
                out = model(**enc)
            logits_list.append(out.logits.cpu())

            if compute_loss:
                labels_t = torch.tensor(batch_labels, dtype=torch.float32)
                total_loss += loss_fn(out.logits.cpu(), labels_t).item()
                n_batches += 1

        probs_by_lang[lang] = torch.sigmoid(torch.cat(logits_list)).numpy()
        true_by_lang[lang] = np.stack(label_rows)

    avg_loss = total_loss / max(n_batches, 1) if compute_loss else None
    return probs_by_lang, true_by_lang, avg_loss

## Training loop

In [ ]:
training_start = time.perf_counter()
history = []

train_split_size = len(dataset["train"])
part_size = train_split_size // N_TRAIN_PARTS

for epoch in tqdm(range(NUM_EPOCHS), desc="Epochs"):
    model.train()
    running_loss = 0.0
    current_step_in_epoch = 0
    epoch_start = time.perf_counter()

    for part_idx in range(N_TRAIN_PARTS):
        start_idx = part_idx * part_size
        end_idx = (part_idx + 1) * part_size if part_idx < N_TRAIN_PARTS - 1 else train_split_size

        raw_train_part = dataset["train"].select(range(start_idx, end_idx))

        chunk_preprocess_fn = partial(
            preprocess_function_multilingual,
            tokenizer=tokenizer, label_to_index=label_to_index, max_length=MAX_LENGTH,
            offset=start_idx, epoch=epoch,
        )

        tokenized_part = raw_train_part.map(
            chunk_preprocess_fn, batched=True, with_indices=True,
            remove_columns=[TEXT_COLUMN, LABEL_COLUMN, "celex_id"],
            load_from_cache_file=False,
        )

        # shuffle=False keeps batch order deterministic/reproducible; the language
        # rotation itself is fixed during .map() above, independent of this.
        train_loader_part = DataLoader(
            tokenized_part, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator
        )
        prepared_train_loader_part = accelerator.prepare(train_loader_part)

        for batch in tqdm(prepared_train_loader_part,
                           desc=f"Epoch {epoch+1} part {part_idx+1}/{N_TRAIN_PARTS}", leave=False):
            optimizer.zero_grad()
            batch["labels"] = batch["labels"].to(torch.float32)

            outputs = model(**batch)
            loss = outputs.loss

            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            current_step_in_epoch += 1

        del raw_train_part, tokenized_part, train_loader_part, prepared_train_loader_part
        torch.cuda.empty_cache()
        gc.collect()

    train_loss = running_loss / max(current_step_in_epoch, 1)

    val_probs_by_lang, val_true_by_lang, val_loss = evaluate_multilingual_full(
        accelerator.unwrap_model(model), tokenizer, dataset["validation"],
        label_to_index, NUM_LABELS, VAL_LANGUAGES, batch_size=BATCH_SIZE, max_length=MAX_LENGTH
    )
    val_pooled_probs = np.concatenate(list(val_probs_by_lang.values()))
    val_pooled_true = np.concatenate(list(val_true_by_lang.values()))
    val_pooled_preds = (val_pooled_probs >= THRESHOLD).astype(int)
    val_metrics = multilabel_metrics(val_pooled_true, val_pooled_preds)
    val_metrics["loss"] = val_loss

    epoch_time = time.perf_counter() - epoch_start

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_micro_f1": val_metrics["micro_f1"],
        "val_macro_f1": val_metrics["macro_f1"],
        "epoch_time_sec": epoch_time,
    })

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | train loss={train_loss:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | val micro-F1={val_metrics['micro_f1']:.4f} | "
        f"val macro-F1={val_metrics['macro_f1']:.4f} | time={epoch_time:.1f}s"
    )

total_training_time = time.perf_counter() - training_start
print(f"Total training time: {total_training_time/60:.2f} minutes")

## Training curves

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training vs Validation Loss"); plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["val_micro_f1"], marker="o", label="Validation micro-F1")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="Validation macro-F1")
plt.xlabel("Epoch"); plt.ylabel("F1"); plt.title("Validation F1 by Epoch"); plt.legend()
plt.show()

## Multilingual evaluation (all 23 languages)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

_inf_start = time.perf_counter()
probs_by_lang, true_by_lang, _ = evaluate_multilingual_full(
    accelerator.unwrap_model(model), tokenizer, dataset["test"],
    label_to_index, NUM_LABELS, LANGUAGES, compute_loss=False
)
inference_time_multilingual = time.perf_counter() - _inf_start

np.savez(
    "/content/drive/MyDrive/XLM-R_MTM_CYCLIC_UNW_eval_probs_by_lang.npz",
    **{f"{l}_probs": probs_by_lang[l] for l in probs_by_lang},
    **{f"{l}_true": true_by_lang[l] for l in true_by_lang}
)
print("Saved raw probabilities + labels to Drive.")

## Per-language metrics, pooled and averaged overall metrics

In [ ]:
def build_per_lang_df(probs_by_lang, true_by_lang, threshold=0.5):
    results = []
    for lang, probs in probs_by_lang.items():
        preds = (probs >= threshold).astype(int)
        y_true = true_by_lang[lang]
        results.append({
            "language": lang,
            "n_examples": len(probs),
            "micro_precision": precision_score(y_true, preds, average="micro", zero_division=0),
            "micro_recall": recall_score(y_true, preds, average="micro", zero_division=0),
            "micro_f1": f1_score(y_true, preds, average="micro", zero_division=0),
            "macro_precision": precision_score(y_true, preds, average="macro", zero_division=0),
            "macro_recall": recall_score(y_true, preds, average="macro", zero_division=0),
            "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        })
    return pd.DataFrame(results)

per_lang_df = build_per_lang_df(probs_by_lang, true_by_lang, threshold=THRESHOLD)
print("=== Per-language metrics (threshold = 0.5) ===")
display(per_lang_df.round(4))

pooled_probs = np.concatenate(list(probs_by_lang.values()))
pooled_true = np.concatenate(list(true_by_lang.values()))
pooled_preds = (pooled_probs >= THRESHOLD).astype(int)

pooled_overall = {
    "micro_precision": precision_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "micro_recall": recall_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "micro_f1": f1_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "macro_precision": precision_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "macro_recall": recall_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "macro_f1": f1_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "exact_match": accuracy_score(pooled_true, pooled_preds),
}
print("\n=== Overall metrics — POOLED across all 23 languages ===")
display(pd.Series(pooled_overall).round(4))

overall_avg = per_lang_df[[
    "micro_precision", "micro_recall", "micro_f1",
    "macro_precision", "macro_recall", "macro_f1"
]].mean()
print("\n=== Overall metrics — AVERAGED across the 23 per-language scores ===")
display(overall_avg.round(4))

per_lang_df.to_csv("/content/drive/MyDrive/langF1(XLM-R)M-T-M_CYCLIC_UNW.csv", index=False)
print("Saved per-language metrics to Drive.")

## Inference time

In [ ]:
n_inference_examples = sum(len(p) for p in probs_by_lang.values())
print(f"Inference examples (pooled, 23 languages): {n_inference_examples}")
print(f"Total inference time: {inference_time_multilingual:.2f} sec")
print(f"Documents/sec: {n_inference_examples / inference_time_multilingual:.2f}")

## Model size

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")